# Matrix Factorization

Companion notebook for the [Matrix Factorization lesson](https://ml-viz.vercel.app/courses/recommender-systems/02-matrix-factorization).

We implement matrix factorization **by SGD** on the observed entries of a sparse rating matrix,
watch it fill in the unknown ratings, and confirm that a low rank reconstructs a low-rank matrix
while regularization controls overfitting. Pure NumPy.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive.

In [ ]:
import numpy as np
np.set_printoptions(precision=2, suppress=True)
rng = np.random.default_rng(0)

## 1 — A sparse, genuinely low-rank rating matrix

We generate a rank-2 'true' matrix from hidden factors, then hide most entries (mask) — exactly the
recommender setup: a few latent tastes generated the ratings, but we only observe a fraction.

In [ ]:
n_users, n_items, true_k = 20, 15, 2
P_true = rng.normal(size=(n_users, true_k))
Q_true = rng.normal(size=(n_items, true_k))
R_full = P_true @ Q_true.T                      # the complete (unknown) ratings

mask = rng.random((n_users, n_items)) < 0.4     # observe ~40% of entries
print(f'observed {mask.sum()} of {n_users*n_items} ratings ({100*mask.mean():.0f}%)')

## 2 — Fit factors by SGD on observed entries only

For each observed (u,i) we step P[u] and Q[i] to reduce the squared error, with L2 regularization.
Critically, we never touch the unobserved entries during training.

In [ ]:
def train_mf(R, mask, k=2, lr=0.02, reg=0.05, epochs=200, seed=0):
    r = np.random.default_rng(seed)
    n, m = R.shape
    P = r.normal(scale=0.1, size=(n, k))
    Q = r.normal(scale=0.1, size=(m, k))
    obs = list(zip(*np.where(mask)))
    for _ in range(epochs):
        r.shuffle(obs)
        for u, i in obs:
            err = R[u, i] - P[u] @ Q[i]
            pu = P[u].copy()
            P[u] += lr * (err * Q[i] - reg * P[u])
            Q[i] += lr * (err * pu  - reg * Q[i])
    return P, Q

P, Q = train_mf(R_full, mask, k=2)
R_hat = P @ Q.T

obs_rmse = np.sqrt(((R_full - R_hat)[mask] ** 2).mean())
hid_rmse = np.sqrt(((R_full - R_hat)[~mask] ** 2).mean())
print(f'RMSE on observed  ratings: {obs_rmse:.3f}')
print(f'RMSE on HIDDEN    ratings: {hid_rmse:.3f}  <- the real test: did it generalize?')

## 3 — Rank controls under/overfitting

The true rank is 2. Too small a k can't capture the structure; too large overfits the observed
entries and predicts the hidden ones worse. We sweep k and watch held-out (hidden) RMSE.

In [ ]:
for k in [1, 2, 3, 5, 8]:
    P, Q = train_mf(R_full, mask, k=k, epochs=200)
    hid = np.sqrt(((R_full - P @ Q.T)[~mask] ** 2).mean())
    print(f'k={k}: hidden-rating RMSE = {hid:.3f}')
print('\nBest generalization is near the true rank (k=2).')

## ✏️ Your turn

**Exercise.** Implement `predict_rating(P, Q, u, i)` (the latent-factor dot product) and
`rmse_on_mask(R, R_hat, mask)` (root-mean-square error over only the entries where `mask` is True).
These are the two primitives the trainer and evaluation above rely on.

In [ ]:
def predict_rating(P, Q, u, i):
    # TODO(you): the predicted rating is the dot product of user u's and item i's factor vectors
    return ...

def rmse_on_mask(R, R_hat, mask):
    # TODO(you): RMSE computed only over entries where mask is True
    return ...

In [ ]:
# This assert cell passes silently when your implementation is correct.
P, Q = train_mf(R_full, mask, k=2)
R_hat = P @ Q.T
assert np.isclose(predict_rating(P, Q, 3, 4), P[3] @ Q[4])
assert np.isclose(rmse_on_mask(R_full, R_hat, mask),
                  np.sqrt(((R_full - R_hat)[mask] ** 2).mean()))
# a rank-2 fit recovers a rank-2 matrix well on held-out entries
assert rmse_on_mask(R_full, R_hat, ~mask) < 0.5
print('\u2713 prediction and masked-RMSE are correct')

<details>
<summary>Solution</summary>

```python
def predict_rating(P, Q, u, i):
    return P[u] @ Q[i]

def rmse_on_mask(R, R_hat, mask):
    return np.sqrt(((R - R_hat)[mask] ** 2).mean())
```

The dot product *is* the model. Training only on observed entries and checking RMSE on the hidden
ones is the recommender analogue of a train/test split — it measures whether the latent factors
generalize to ratings the model never saw, which is the whole goal.

</details>